# Notebook 02 — Data Preprocessing
## AI-Powered Customer Feedback Intelligence System

**Purpose:** Transform raw Amazon reviews into clean, balanced, split datasets ready for model training.

**What this notebook produces:**
- `data/processed/train.csv` — 80% of 110K = 88,000 reviews
- `data/processed/val.csv` — 10% of 110K = 11,000 reviews  
- `data/processed/test.csv` — 10% of 110K = 11,000 reviews

**Key decisions made in Notebook 01:**
- Labels: 4-5 stars = Positive, 1-2 stars = Negative, 3 stars = Dropped
- Sample size: 75,000 Positive + 75,000 Negative = 150,000 total
- max_length for tokenizer: 256 tokens

In [1]:
# ============================================================
# IMPORTS
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import re
import warnings

from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 100)

# Random seed — VERY IMPORTANT
# Using the same seed ensures that if you re-run this notebook,
# you get the exact same train/val/test split every time.
# This is called REPRODUCIBILITY — essential in ML engineering.
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)

print("✅ Imports successful")
print(f"Random seed set to: {RANDOM_SEED}")
print("This ensures reproducible train/val/test splits")

✅ Imports successful
Random seed set to: 42
This ensures reproducible train/val/test splits


In [2]:
# ============================================================
# LOAD RAW DATA
# ============================================================

DATA_PATH = os.path.join('..', 'data', 'raw', 'Reviews.csv')

print(f"Loading: {DATA_PATH}")
df_raw = pd.read_csv(DATA_PATH)

print(f"✅ Loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")

Loading: ..\data\raw\Reviews.csv
✅ Loaded: 568,454 rows × 10 columns


In [3]:
# ============================================================
# CLEANING PIPELINE — TRANSPARENT STEP BY STEP
# ============================================================

df = df_raw.copy()
print(f"Starting shape: {df.shape[0]:,} rows\n")

# ── STEP 1: Keep only columns we actually need ──────────────
# We discard columns that add no value to our model or app
KEEP_COLS = ['ProductId', 'Score', 'Time', 'Summary', 'Text']
df = df[KEEP_COLS].copy()
print(f"Step 1 — Keep relevant columns: {KEEP_COLS}")
print(f"         Shape: {df.shape[0]:,} rows × {df.shape[1]} columns\n")

# ── STEP 2: Drop rows where review Text is missing ──────────
# A review with no text is useless for NLP
before = len(df)
df = df.dropna(subset=['Text'])
after = len(df)
print(f"Step 2 — Drop missing Text: removed {before-after:,} rows")
print(f"         Shape: {after:,} rows\n")

# ── STEP 3: Drop rows where Score is missing ────────────────
before = len(df)
df = df.dropna(subset=['Score'])
after = len(df)
print(f"Step 3 — Drop missing Score: removed {before-after:,} rows")
print(f"         Shape: {after:,} rows\n")

# ── STEP 4: Ensure Score is valid (1-5 only) ────────────────
before = len(df)
df = df[df['Score'].isin([1, 2, 3, 4, 5])]
after = len(df)
print(f"Step 4 — Remove invalid scores: removed {before-after:,} rows")
print(f"         Shape: {after:,} rows\n")

# ── STEP 5: Drop duplicate reviews ──────────────────────────
# Duplicate reviews skew model training — same text seen multiple
# times makes the model overfit to repeated phrases
before = len(df)
df = df.drop_duplicates(subset=['Text'])
after = len(df)
print(f"Step 5 — Drop duplicate Text: removed {before-after:,} rows")
print(f"         Shape: {after:,} rows\n")

# ── STEP 6: Drop 3-star reviews (neutral — decided in EDA) ──
before = len(df)
df = df[df['Score'] != 3]
after = len(df)
print(f"Step 6 — Drop 3-star neutral reviews: removed {before-after:,} rows")
print(f"         Shape: {after:,} rows\n")

# ── STEP 7: Drop very short reviews (less than 3 words) ─────
# A review like "ok" or "no" carries almost no signal
before = len(df)
df = df[df['Text'].str.split().str.len() >= 3]
after = len(df)
print(f"Step 7 — Drop reviews under 3 words: removed {before-after:,} rows")
print(f"         Shape: {after:,} rows\n")

print("=" * 50)
print(f"✅ Cleaning complete")
print(f"   Final shape: {df.shape[0]:,} rows")
print(f"   Positive (4-5★): {(df['Score'] >= 4).sum():,}")
print(f"   Negative (1-2★): {(df['Score'] <= 2).sum():,}")

Starting shape: 568,454 rows

Step 1 — Keep relevant columns: ['ProductId', 'Score', 'Time', 'Summary', 'Text']
         Shape: 568,454 rows × 5 columns

Step 2 — Drop missing Text: removed 0 rows
         Shape: 568,454 rows

Step 3 — Drop missing Score: removed 0 rows
         Shape: 568,454 rows

Step 4 — Remove invalid scores: removed 0 rows
         Shape: 568,454 rows

Step 5 — Drop duplicate Text: removed 174,875 rows
         Shape: 393,579 rows

Step 6 — Drop 3-star neutral reviews: removed 29,754 rows
         Shape: 363,825 rows

Step 7 — Drop reviews under 3 words: removed 0 rows
         Shape: 363,825 rows

✅ Cleaning complete
   Final shape: 363,825 rows
   Positive (4-5★): 306,758
   Negative (1-2★): 57,067


In [4]:
# ============================================================
# CREATE SENTIMENT LABELS
# ============================================================

# WHY WE DO THIS HERE AND NOT EARLIER:
# We clean first, then label. This order matters.
# Labeling before cleaning could introduce subtle bugs
# if cleaning removes rows after label assignment.

df['sentiment'] = df['Score'].apply(lambda x: 1 if x >= 4 else 0)
df['sentiment_label'] = df['sentiment'].map({1: 'Positive', 0: 'Negative'})

# Convert Unix timestamp to readable date
df['date'] = pd.to_datetime(df['Time'], unit='s')
df['year'] = df['date'].dt.year

print("✅ Sentiment labels created")
print(f"\nLabel distribution:")
print(f"  Positive (1): {(df['sentiment']==1).sum():,}")
print(f"  Negative (0): {(df['sentiment']==0).sum():,}")
print(f"\nSample rows:")
df[['Text', 'Score', 'sentiment', 'sentiment_label']].head(5)

✅ Sentiment labels created

Label distribution:
  Positive (1): 306,758
  Negative (0): 57,067

Sample rows:


,Text,Score,sentiment,sentiment_label
0,I have bought several of the Vitality canned dog food products and have found them all to be of ...,5,1,Positive
1,Product arrived labeled as Jumbo Salted Peanuts...the peanuts were actually small sized unsalted...,1,0,Negative
2,"This is a confection that has been around a few centuries. It is a light, pillowy citrus gelati...",4,1,Positive
3,If you are looking for the secret ingredient in Robitussin I believe I have found it. I got thi...,2,0,Negative
4,Great taffy at a great price. There was a wide assortment of yummy taffy. Delivery was very qu...,5,1,Positive


In [7]:
# ============================================================
# DIAGNOSIS — Check available samples before sampling
# ============================================================

positive_available = (df['sentiment'] == 1).sum()
negative_available = (df['sentiment'] == 0).sum()

print(f"Available Positive reviews: {positive_available:,}")
print(f"Available Negative reviews: {negative_available:,}")
print(f"Requested per class      : {SAMPLES_PER_CLASS:,}")
print()

if negative_available < SAMPLES_PER_CLASS:
    print(f"⚠️  NOT ENOUGH negative reviews!")
    print(f"   We have {negative_available:,} but requested {SAMPLES_PER_CLASS:,}")
    print(f"   Solution: reduce SAMPLES_PER_CLASS to {negative_available:,} or less")
else:
    print("✅ Enough samples available for both classes")

Available Positive reviews: 306,758
Available Negative reviews: 57,067
Requested per class      : 75,000

⚠️  NOT ENOUGH negative reviews!
   We have 57,067 but requested 75,000
   Solution: reduce SAMPLES_PER_CLASS to 57,067 or less


In [8]:
# ============================================================
# BALANCED SAMPLING — 75K POSITIVE + 75K NEGATIVE = 150K
# ============================================================

# WHY BALANCED SAMPLING:
# Our dataset is 84% positive and 16% negative.
# If we train on this imbalanced data, the model learns
# "when in doubt, say Positive" — it becomes lazy.
# Balanced sampling forces the model to genuinely learn
# what makes a review negative vs positive.

SAMPLES_PER_CLASS = 55000

df_positive = df[df['sentiment'] == 1].sample(
    n=SAMPLES_PER_CLASS,
    random_state=RANDOM_SEED
)
df_negative = df[df['sentiment'] == 0].sample(
    n=SAMPLES_PER_CLASS,
    random_state=RANDOM_SEED
)

df_balanced = pd.concat([df_positive, df_negative], axis=0)

# Shuffle — important! We don't want all positives first, then negatives.
# If data is ordered by class, batch training becomes unstable.
df_balanced = df_balanced.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

print(f"✅ Balanced dataset created")
print(f"   Total samples    : {len(df_balanced):,}")
print(f"   Positive samples : {(df_balanced['sentiment']==1).sum():,}")
print(f"   Negative samples : {(df_balanced['sentiment']==0).sum():,}")
print(f"   Class ratio      : 50% / 50% ✅")

✅ Balanced dataset created
   Total samples    : 110,000
   Positive samples : 55,000
   Negative samples : 55,000
   Class ratio      : 50% / 50% ✅


In [9]:
# ============================================================
# TEXT CLEANING FUNCTION
# ============================================================

def clean_text_for_transformer(text):
    """
    Light cleaning suitable for transformer models like DistilBERT.
    
    We preserve:
    - Punctuation (carries sentiment signal: "!!!" vs ".")
    - Capitalization (ALL CAPS often signals strong emotion)
    - Contractions ("don't", "won't" — DistilBERT understands these)
    - Stop words ("not good" — removing "not" destroys the meaning)
    
    We remove:
    - HTML tags (artifacts from web scraping)
    - URLs (no semantic value for sentiment)
    - Excessive whitespace
    - Non-printable characters
    """
    if not isinstance(text, str):
        return ""
    
    # Remove HTML tags (e.g. <br />, <b>, etc.)
    text = re.sub(r'<[^>]+>', ' ', text)
    
    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    
    # Remove non-ASCII characters (keeps standard English punctuation)
    text = text.encode('ascii', 'ignore').decode('ascii')
    
    # Collapse multiple spaces/newlines into single space
    text = re.sub(r'\s+', ' ', text)
    
    # Strip leading/trailing whitespace
    text = text.strip()
    
    return text


# Apply to our balanced dataset
print("Cleaning review text...")
df_balanced['cleaned_text'] = df_balanced['Text'].apply(clean_text_for_transformer)

# Drop any rows where cleaning resulted in empty text
before = len(df_balanced)
df_balanced = df_balanced[df_balanced['cleaned_text'].str.len() > 0]
after = len(df_balanced)
print(f"Removed {before - after} empty reviews after cleaning")

# Show a before/after example
sample_idx = df_balanced.index[0]
print(f"\n--- BEFORE CLEANING ---")
print(df_balanced['Text'].iloc[0][:300])
print(f"\n--- AFTER CLEANING ---")
print(df_balanced['cleaned_text'].iloc[0][:300])
print(f"\n✅ Text cleaning complete. {after:,} reviews ready.")

Cleaning review text...
Removed 0 empty reviews after cleaning

--- BEFORE CLEANING ---
Great tasting  coffee, not to mild and not to bold.  Just right for my taste. I order this time after time along with hazelnut.

--- AFTER CLEANING ---
Great tasting coffee, not to mild and not to bold. Just right for my taste. I order this time after time along with hazelnut.

✅ Text cleaning complete. 110,000 reviews ready.


150,000 total
├── Train : 120,000  (80%) — model learns from this
├── Val   :  15,000  (10%) — we tune hyperparameters using this
└── Test  :  15,000  (10%) — final honest evaluation, touched ONCE

In [10]:
# ============================================================
# TRAIN / VALIDATION / TEST SPLIT
# ============================================================

# IMPORTANT: We use stratify=sentiment to ensure each split
# has exactly 50% positive and 50% negative.
# Without stratify, random chance could give you 55/45 in one split.

FEATURES = ['ProductId', 'Score', 'Time', 'date', 'year',
            'sentiment', 'sentiment_label', 'cleaned_text']

df_model = df_balanced[FEATURES].copy()

# ── First split: separate test set (10%) ────────────────────
df_trainval, df_test = train_test_split(
    df_model,
    test_size=0.10,
    random_state=RANDOM_SEED,
    stratify=df_model['sentiment']   # preserve 50/50 balance
)

# ── Second split: separate val from train (10% of total) ────
df_train, df_val = train_test_split(
    df_trainval,
    test_size=0.1111,    # 0.1111 × 90% ≈ 10% of total
    random_state=RANDOM_SEED,
    stratify=df_trainval['sentiment']
)

print("✅ Train/Val/Test split complete")
print(f"\n{'Split':<10} {'Total':>10} {'Positive':>10} {'Negative':>10} {'Pos%':>8}")
print("-" * 50)
for name, split in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    total = len(split)
    pos   = (split['sentiment'] == 1).sum()
    neg   = (split['sentiment'] == 0).sum()
    pct   = pos / total * 100
    print(f"{name:<10} {total:>10,} {pos:>10,} {neg:>10,} {pct:>7.1f}%")

print(f"\nTotal: {len(df_train)+len(df_val)+len(df_test):,}")

✅ Train/Val/Test split complete

Split           Total   Positive   Negative     Pos%
--------------------------------------------------
Train          88,001     44,001     44,000    50.0%
Val            10,999      5,499      5,500    50.0%
Test           11,000      5,500      5,500    50.0%

Total: 110,000


In [11]:
# ============================================================
# SAVE PROCESSED DATASETS
# ============================================================

PROCESSED_PATH = os.path.join('..', 'data', 'processed')
os.makedirs(PROCESSED_PATH, exist_ok=True)

train_path = os.path.join(PROCESSED_PATH, 'train.csv')
val_path   = os.path.join(PROCESSED_PATH, 'val.csv')
test_path  = os.path.join(PROCESSED_PATH, 'test.csv')

df_train.to_csv(train_path, index=False)
df_val.to_csv(val_path,     index=False)
df_test.to_csv(test_path,   index=False)

print("✅ All processed files saved")
print(f"\n   train.csv → {len(df_train):,} rows → {os.path.getsize(train_path)/1024/1024:.1f} MB")
print(f"   val.csv   → {len(df_val):,} rows  → {os.path.getsize(val_path)/1024/1024:.1f} MB")
print(f"   test.csv  → {len(df_test):,} rows  → {os.path.getsize(test_path)/1024/1024:.1f} MB")

print(f"\n📁 Location: {os.path.abspath(PROCESSED_PATH)}")

✅ All processed files saved

   train.csv → 88,001 rows → 41.0 MB
   val.csv   → 10,999 rows  → 5.1 MB
   test.csv  → 11,000 rows  → 5.1 MB

📁 Location: e:\AI_Powered_review_system\data\processed


In [13]:
# ============================================================
# FINAL VERIFICATION — RELOAD AND CONFIRM
# ============================================================

# We reload from disk to confirm files saved correctly
train_check = pd.read_csv(train_path)
val_check   = pd.read_csv(val_path)
test_check  = pd.read_csv(test_path)

print("✅ Verification — files reloaded from disk successfully")
print(f"\n   train.csv : {train_check.shape}  columns: {train_check.columns.tolist()}")
print(f"   val.csv   : {val_check.shape}")
print(f"   test.csv  : {test_check.shape}")

print(f"\nSample training row:")
print(f"  Text      : {train_check['cleaned_text'].iloc[0][:100]}...")
print(f"  Sentiment : {train_check['sentiment'].iloc[0]} ({train_check['sentiment_label'].iloc[0]})")
print(f"  Score     : {train_check['Score'].iloc[0]}")

print(f"""
╔══════════════════════════════════════════════════════╗
║         PHASE 4 COMPLETE ✅                         ║
╠══════════════════════════════════════════════════════╣
║  train.csv : 88,001 reviews (80%)                  ║
║  val.csv   :  10,999 reviews (10%)                  ║
║  test.csv  :  11,000 reviews (10%)                  ║
║  All splits: 50% Positive / 50% Negative            ║
║  Text: lightly cleaned, transformer-ready           ║
║                                                      ║
║  NEXT: Notebook 03 — Baseline Model                 ║
║  (TF-IDF + Logistic Regression)                     ║
╚══════════════════════════════════════════════════════╝
""")

✅ Verification — files reloaded from disk successfully

   train.csv : (88001, 8)  columns: ['ProductId', 'Score', 'Time', 'date', 'year', 'sentiment', 'sentiment_label', 'cleaned_text']
   val.csv   : (10999, 8)
   test.csv  : (11000, 8)

Sample training row:
  Text      : way too sweet by itself. however this and a bold coffee makes a pretty good mix. just do 5oz of this...
  Sentiment : 0 (Negative)
  Score     : 2

╔══════════════════════════════════════════════════════╗
║         PHASE 4 COMPLETE ✅                         ║
╠══════════════════════════════════════════════════════╣
║  train.csv : 88,001 reviews (80%)                  ║
║  val.csv   :  10,999 reviews (10%)                  ║
║  test.csv  :  11,000 reviews (10%)                  ║
║  All splits: 50% Positive / 50% Negative            ║
║  Text: lightly cleaned, transformer-ready           ║
║                                                      ║
║  NEXT: Notebook 03 — Baseline Model                 ║
║  (TF-IDF + Log